Check silver table

In [0]:
# Check Silver tables

spark.sql("SHOW TABLES IN traveldbs.silverdbs").show(
    truncate=False
)

+---------+-------------------------+-----------+
|database |tableName                |isTemporary|
+---------+-------------------------+-----------+
|silverdbs|customers                |false      |
|silverdbs|delivery_events          |false      |
|silverdbs|driver_monthly_metrics   |false      |
|silverdbs|drivers                  |false      |
|silverdbs|facilities               |false      |
|silverdbs|fuel_purchases           |false      |
|silverdbs|loads                    |false      |
|silverdbs|maintenance_records      |false      |
|silverdbs|routes                   |false      |
|silverdbs|safety_incidents         |false      |
|silverdbs|trailers                 |false      |
|silverdbs|trips                    |false      |
|silverdbs|truck_utilization_metrics|false      |
|silverdbs|trucks                   |false      |
+---------+-------------------------+-----------+



Create Gold Database

In [0]:
# Create Gold database

spark.sql("""
CREATE DATABASE IF NOT EXISTS traveldbs.golddb
""")

print("Gold database created successfully")

Gold database created successfully


Create Gold tables

In [0]:
spark.sql("DROP TABLE IF EXISTS traveldbs.golddb.driver_performance")

DataFrame[]

In [0]:
from pyspark.sql import functions as F

# Read Silver driver monthly metrics
driver_metrics = spark.table(
    "traveldbs.silverdbs.driver_monthly_metrics"
)

# Create Driver Performance analysis
driver_performance = (
    driver_metrics
    .groupBy("driver_id")
    .agg(
        F.count("*").alias("total_records")
    )
)

# Save as Gold Delta table
driver_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "traveldbs.golddb.driver_performance"
    )

display(driver_performance)

print("Driver performance Gold table created successfully")

driver_id,total_records
DRV00001,36
DRV00002,36
DRV00003,36
DRV00005,36
DRV00006,36
DRV00009,36
DRV00011,36
DRV00012,36
DRV00013,36
DRV00014,36


Driver performance Gold table created successfully


Route Profitability

In [0]:
routes = spark.table("traveldbs.silverdbs.routes")
loads = spark.table("traveldbs.silverdbs.loads")

route_profitability = (
    routes
    .join(loads, "route_id", "left")
)

route_profitability.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("traveldbs.golddb.route_profitability")

print("route_profitability created")

route_profitability created


Fleet Utilization

In [0]:
trucks = spark.table("traveldbs.silverdbs.trucks")
truck_util = spark.table(
    "traveldbs.silverdbs.truck_utilization_metrics"
)

fleet_utilization = (
    trucks
    .join(truck_util, "truck_id", "left")
)

fleet_utilization.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("traveldbs.golddb.fleet_utilization")

print("fleet_utilization created")

fleet_utilization created


Maintenance Analysis

In [0]:
from pyspark.sql import functions as F

# Read Silver table
maintenance = spark.table(
    "traveldbs.silverdbs.maintenance_records"
)

# Create Gold analysis
maintenance_analysis = (
    maintenance
    .groupBy("truck_id")
    .agg(
        F.count("*").alias("maintenance_count")
    )
)

# Save Gold table
maintenance_analysis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "traveldbs.golddb.maintenance_analysis"
    )

print("Maintenance analysis Gold table created successfully")

Maintenance analysis Gold table created successfully


Fuel Efficiency

In [0]:
from pyspark.sql import functions as F

# Read Silver table
fuel = spark.table(
    "traveldbs.silverdbs.fuel_purchases"
)

# Fuel Efficiency Analysis
fuel_efficiency = (
    fuel
    .groupBy("truck_id")
    .agg(
        F.round(
            F.sum("gallons"), 2
        ).alias("total_fuel")
    )
)

# Save Gold table
fuel_efficiency.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "traveldbs.golddb.fuel_efficiency"
    )

display(fuel_efficiency)

truck_id,total_fuel
TRK00057,272706.9
TRK00101,258214.7
TRK00115,271910.6
TRK00001,279468.1
TRK00090,259308.1
TRK00059,243604.1
TRK00091,258063.0
TRK00040,240873.4
TRK00085,262838.1
TRK00015,273774.3


In [0]:
display(
    spark.table(
        "traveldbs.golddb.fuel_efficiency"
    )
)

truck_id,total_fuel
TRK00052,247033.3
TRK00018,243048.8
TRK00025,281631.0
TRK00070,262426.4
TRK00098,260716.9
TRK00016,261531.5
TRK00076,259011.3
TRK00064,269412.9
TRK00020,269945.8
TRK00080,252715.9


Driver Performance

In [0]:
from pyspark.sql import functions as F
# Read Silver table
driver = spark.table(
    "traveldbs.silverdbs.driver_monthly_metrics"
)

# Create Driver Performance analysis
driver_performance = (
    driver
    .groupBy("driver_id")
    .agg(
        F.count("*").alias("total_records")
    )
)

# Save as Gold Delta table
driver_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "traveldbs.golddb.driver_performance"
    )

# Display result
display(driver_performance)

print("Driver Performance Gold table created successfully")

driver_id,total_records
DRV00001,36
DRV00002,36
DRV00003,36
DRV00005,36
DRV00006,36
DRV00009,36
DRV00011,36
DRV00012,36
DRV00013,36
DRV00014,36


Driver Performance Gold table created successfully


Customer analysis

In [0]:
from pyspark.sql import functions as F

# ==============================
# GOLD - CUSTOMER ANALYSIS
# ==============================

# Read Silver customers table
customers = spark.table(
    "traveldbs.silverdbs.customers"
)

# Create Customer Analysis
customer_analysis = (
    customers
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("total_records")
    )
)

# Save as Gold Delta table
customer_analysis.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "traveldbs.golddb.customer_analysis"
    )

display(customer_analysis)

print("Customer Analysis Gold table created successfully")

customer_id,total_records
CUST00040,1
CUST00056,1
CUST00087,1
CUST00089,1
CUST00093,1
CUST00100,1
CUST00105,1
CUST00134,1
CUST00146,1
CUST00151,1


Customer Analysis Gold table created successfully


Safety Metrics

In [0]:
from pyspark.sql import functions as F
# Read Silver safety incidents table
safety = spark.table(
    "traveldbs.silverdbs.safety_incidents"
)

# Create Safety Metrics
safety_metrics = (
    safety
    .groupBy("truck_id")
    .agg(
        F.count("*").alias("incident_count")
    )
)

# Save as Gold Delta table
safety_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "traveldbs.golddb.safety_metrics"
    )

display(safety_metrics)

print("Safety Metrics Gold table created successfully")

truck_id,incident_count
TRK00074,2
TRK00108,2
TRK00093,2
TRK00040,3
TRK00006,2
TRK00057,4
TRK00004,2
TRK00120,2
TRK00062,4
TRK00101,1


Safety Metrics Gold table created successfully


Seasonal Patterns

In [0]:
from pyspark.sql import functions as F

# Read Silver delivery events
events = spark.table(
    "traveldbs.silverdbs.delivery_events"
)

# Create Seasonal Patterns
seasonal_patterns = (
    events
    .withColumn(
        "month",
        F.month("scheduled_datetime")
    )
    .groupBy("month")
    .agg(
        F.count("*").alias("total_events")
    )
    .orderBy("month")
)

# Save as Gold Delta table
seasonal_patterns.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "traveldbs.golddb.seasonal_patterns"
    )

display(seasonal_patterns)

print("Seasonal Patterns Gold table created successfully")

month,total_events
1,14597
2,13191
3,14391
4,14105
5,14437
6,14185
7,14489
8,14578
9,13919
10,14687


Seasonal Patterns Gold table created successfully


Validation

In [0]:
gold_tables = [
    "driver_performance",
    "route_profitability",
    "fleet_utilization",
    "maintenance_analysis",
    "fuel_efficiency",
    "customer_analysis",
    "safety_metrics",
    "seasonal_patterns"
]

for table in gold_tables:
    df = spark.table(f"traveldbs.golddb.{table}")
    print(f"{table}: {df.count()} rows")

driver_performance: 124 rows
route_profitability: 85410 rows
fleet_utilization: 3340 rows
maintenance_analysis: 120 rows
fuel_efficiency: 93 rows
customer_analysis: 200 rows
safety_metrics: 81 rows
seasonal_patterns: 12 rows
